In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pooled = pd.read_csv("<PATH_TO_BP_PRE_POST_TSV>", sep = "\t")
split = pd.read_csv("<PATH_TO_BP_PRE_POST_AGE5_WITH_STATINS_TSV>", sep = "\t")

In [ ]:
bed_dir = "<PATH_TO_MERGED_BEDFILES_DIR>"
chrom = 1
fam_path = f"{bed_dir}/c{chrom}.fam"

iids = set(pd.read_csv(fam_path, sep=r"\s+", header=None, usecols=[1])[1].astype(str))

df_pooled = pooled[pooled["eid"].astype(str).isin(iids)].copy()
df_strat  = split[split["eid"].astype(str).isin(iids)].copy()

def is_binary(s):
    u = pd.unique(s.dropna())
    return len(u) > 0 and set(u).issubset({0, 1})

def summarize(df, dataset, mask=None):
    d = df if mask is None else df.loc[mask]
    cols = [c for c in d.columns if c != "eid"]
    return pd.DataFrame(
        {
            "sample size": [int(d[c].notna().sum()) for c in cols],
            "trait": cols,
            "dataset": dataset,
            "prevalence": [float(d[c].mean()) if is_binary(d[c]) else np.nan for c in cols],
        }
    )

med_cols = ["ACE_inhibitor","angiotensin_receptor_blocker","calcium_channel_blocker",
            "beta_blocker","diuretic","statin"]
on_drugs = df_pooled[med_cols].fillna(0).sum(axis=1).gt(0)

summary = pd.concat(
    [
        summarize(df_pooled, "age-pooled all"),
        summarize(df_pooled, "age-pooled on drugs", mask=on_drugs),
        summarize(df_strat,  "age-stratified"),
    ],
    ignore_index=True,
).sort_values(["dataset", "trait"], kind="stable").reset_index(drop=True)


In [ ]:
summary.to_csv("<PATH_TO_SAMPLE_SIZE_CSV>")